# Rutas del Bajío · Evidencias del reporte técnico

Este cuaderno genera **todas las capturas y pruebas** que pide el reporte, en el mismo orden del documento. Cada celda imprime un encabezado con el número de figura o tabla y la fecha y hora, para que la captura quede lista para pegarse.

**Cómo usarlo**
1. Menú **Entorno de ejecución → Ejecutar todas**.
2. Toma captura de la salida de cada celda marcada con 📸, con el encabezado y la fecha visibles.
3. Pégala en el lugar del reporte que indica la tabla de abajo.

| Celda | Captura | Va en el reporte |
|---|---|---|
| 2 | Estructura de archivos de pruebas y fixtures | Capítulo 3, Tabla 15 (apoyo) |
| 3 | Reporte final de pytest | Anexo C, **Figura 7** · Tabla 17 |
| 4 | Ejecución por marcador alta | Anexo C, **Figura 8** · Tabla 18 |
| 5 | Ejecución por marcador media | Anexo C, **Figura 9** · Tabla 18 |
| 6 | Lista de cotejo calculada con el resultado real | Anexo C, **Figura 10** · Tabla 19 |
| 7 | Mutaciones reales y marcador baja | Anexo C, **Figura 12** · Tablas 18 y 20 |
| 8 | Cobertura de la matriz de criterios | Capítulo 3, Tabla 16 (apoyo) |
| 9 | Árbol real del repositorio | Capítulo 2, **Figura 4** |
| 10 | Historial y commits por integrante | Anexo D, **Figura 13** · Tabla 24 |

**Lo que este cuaderno no puede generar:** las Figuras 5 y 6 (lote de ejemplo P-001 y rechazos) salen del código del sistema de Luis (`carga_lote.py`), que no está en este cuaderno ni en el repositorio. Las actas, bitácoras y firmas tampoco salen de aquí.

> La hora de Colab va 6 horas adelante de la hora de León (Colab usa UTC).

## 1. Preparación
Función para los encabezados de cada captura.

In [ ]:
import datetime
def encabezado(titulo):
    print("=" * 64)
    print("  " + titulo)
    print("  FECHA Y HORA DE EJECUCIÓN:", datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "(reloj de Colab)")
    print("=" * 64 + "\n")
print("Listo: función encabezado() disponible.")

## 2. Archivos de pruebas · Tabla 15 📸
Escribe los mismos archivos de pruebas del cuaderno del equipo (`Untitled3.ipynb`) y muestra la estructura de la carpeta y las fixtures de `conftest.py`.

In [ ]:
import os
import datetime

# 1. Asegurar la recreación de la estructura de archivos de prueba
os.makedirs("tests", exist_ok=True)

# Escribir el archivo pytest.ini
with open("pytest.ini", "w", encoding="utf-8") as f:
    f.write("[pytest]\nmarkers =\n    alta: Pruebas de prioridad alta\n    media: Pruebas de prioridad media\n    baja: Pruebas de prioridad baja\n")

# Escribir conftest.py
with open("tests/conftest.py", "w", encoding="utf-8") as f:
    f.write('''import pytest
import csv

@pytest.fixture
def datos_ruta_valida():
    return {"id": "R-001", "nombre": "León - Silao", "origen": "León", "destino": "Silao", "costo_base": 150.0}

@pytest.fixture
def archivo_csv_temporal(tmp_path):
    archivo = tmp_path / "rutas_prueba.csv"
    with open(archivo, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "nombre", "origen", "destino", "costo_base"])
        writer.writerow(["R-001", "León - Silao", "León", "Silao", "150.0"])
    return archivo
''')

# Escribir test_modelos.py
with open("tests/test_modelos.py", "w", encoding="utf-8") as f:
    f.write('''import pytest

class Ruta:
    def __init__(self, id, nombre, origen, destino, costo_base):
        if costo_base <= 0:
            raise ValueError("El costo base debe ser mayor a 0")
        self.id = id
        self.nombre = nombre
        self.origen = origen
        self.destino = destino
        self.costo_base = costo_base

class MaquinaEstadosPedido:
    TRANSICIONES_VALIDAS = {
        "Registrado": ["Validado", "Cancelado"],
        "Validado": ["En tránsito", "Cancelado"],
        "En tránsito": ["Entregado"],
        "Entregado": [],
        "Cancelado": []
    }
    def __init__(self):
        self.estado_actual = "Registrado"
    def cambiar_estado(self, nuevo_estado):
        if nuevo_estado in self.TRANSICIONES_VALIDAS.get(self.estado_actual, []):
            self.estado_actual = nuevo_estado
        else:
            raise ValueError(f"Transición inválida")

@pytest.mark.alta
def test_creacion_ruta_valida(datos_ruta_valida):
    ruta = Ruta(**datos_ruta_valida)
    assert ruta.id == "R-001"

@pytest.mark.media
def test_costo_ruta_invalido():
    with pytest.raises(ValueError):
        Ruta("R-002", "León", "L", "I", -50.0)

@pytest.mark.alta
def test_transicion_estado_valida():
    pedido = MaquinaEstadosPedido()
    pedido.cambiar_estado("Validado")
    assert pedido.estado_actual == "Validado"

@pytest.mark.alta
def test_transicion_estado_invalida():
    pedido = MaquinaEstadosPedido()
    with pytest.raises(ValueError):
        pedido.cambiar_estado("Entregado")
''')

# Escribir test_gestores.py
with open("tests/test_gestores.py", "w", encoding="utf-8") as f:
    f.write('''import pytest
import csv

class GestorRutas:
    def __init__(self, archivo_path):
        self.archivo_path = archivo_path
        self.rutas = {}
    def cargar(self):
        if not self.archivo_path.exists(): return
        with open(self.archivo_path, mode="r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rutas[row["id"]] = row
    def agregar(self, ruta_dict):
        if ruta_dict["id"] in self.rutas:
            raise ValueError("ID duplicado")
        self.rutas[ruta_dict["id"]] = ruta_dict

@pytest.mark.alta
def test_cargar_archivo_inexistente(tmp_path):
    archivo = tmp_path / "no_existe.csv"
    gestor = GestorRutas(archivo)
    gestor.cargar()
    assert gestor.rutas == {}

@pytest.mark.alta
def test_agregar_ruta_duplicada(archivo_csv_temporal, datos_ruta_valida):
    gestor = GestorRutas(archivo_csv_temporal)
    gestor.cargar()
    with pytest.raises(ValueError):
        gestor.agregar(datos_ruta_valida)
''')

# Escribir test_carga_lote.py
with open("tests/test_carga_lote.py", "w", encoding="utf-8") as f:
    f.write('''import pytest
import csv

def procesar_lote_rutas(archivo_path):
    procesados, errores = [], []
    with open(archivo_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader, start=2):
            try:
                costo = float(row["costo_base"])
                if costo <= 0: raise ValueError()
                procesados.append(row)
            except Exception as e:
                errores.append({"linea": i, "datos": row, "error": str(e)})
    return procesados, errores

@pytest.mark.media
def test_procesar_lote_con_filas_invalidas(tmp_path):
    archivo = tmp_path / "lote.csv"
    with open(archivo, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "nombre", "origen", "destino", "costo_base"])
        writer.writerow(["R-001", "R1", "A", "B", "100.0"])
        writer.writerow(["R-002", "R2", "A", "B", "-5.0"])
    p, e = procesar_lote_rutas(archivo)
    assert len(p) == 1 and len(e) == 1
''')

encabezado("TABLA 15: ESTRUCTURA DE LA CARPETA DE PRUEBAS")
!find . -path ./sample_data -prune -o -path ./.config -prune -o -name "__pycache__" -prune -o -type f \( -name "*.py" -o -name "*.ini" \) -print | sort
print("\n--- Fixtures en tests/conftest.py ---")
!grep -n -A1 "@pytest.fixture" tests/conftest.py | grep "def "
print("\n--- Pruebas por archivo y su marcador ---")
!grep -n -B1 "^def test_" tests/test_*.py | grep -E "mark|def test_"

## 3. Reporte final de pytest · Figura 7 · Tabla 17 📸
Debe dar **7 passed**.

In [ ]:
encabezado("FIGURA 7 · TABLA 17: REPORTE FINAL DE PRUEBAS UNITARIAS")
!pytest -v

## 4. Marcador alta · Figura 8 · Tabla 18 📸

In [ ]:
encabezado("FIGURA 8 · TABLA 18: EJECUCIÓN POR MARCADOR ALTA")
!pytest -v -m alta

## 5. Marcador media · Figura 9 · Tabla 18 📸

In [ ]:
encabezado("FIGURA 9 · TABLA 18: EJECUCIÓN POR MARCADOR MEDIA")
!pytest -v -m media

## 6. Lista de cotejo · Figura 10 · Tabla 19 📸
A diferencia de la versión anterior, aquí el resultado de cada prueba **se lee de la ejecución real** de pytest (archivo JUnit), no se escribe a mano.

In [ ]:
import subprocess
import xml.etree.ElementTree as ET
import pandas as pd

encabezado("FIGURA 10 · TABLA 19: LISTA DE COTEJO Y PRUEBAS DE ACEPTACIÓN")
subprocess.run(["pytest", "-q", "-p", "no:cacheprovider", "--junitxml=resultado.xml"], capture_output=True)

CRITERIOS = {
    "test_costo_ruta_invalido": ("Ruta (test_modelos.py)", "Validar costo base > 0"),
    "test_creacion_ruta_valida": ("Ruta (test_modelos.py)", "Instanciación correcta con datos válidos"),
    "test_transicion_estado_valida": ("Máquina de estados (test_modelos.py)", "Transición permitida Registrado → Validado"),
    "test_transicion_estado_invalida": ("Máquina de estados (test_modelos.py)", "Bloqueo de la transición Registrado → Entregado"),
    "test_cargar_archivo_inexistente": ("GestorRutas (test_gestores.py)", "Archivo inexistente deja la lista vacía"),
    "test_agregar_ruta_duplicada": ("GestorRutas (test_gestores.py)", "Rechazo de id duplicado"),
    "test_procesar_lote_con_filas_invalidas": ("Procesamiento de lote (test_carga_lote.py)", "Exclusión de filas con costo inválido"),
}
filas = []
for caso in ET.parse("resultado.xml").getroot().iter("testcase"):
    nombre = caso.get("name")
    fallo = caso.find("failure") is not None or caso.find("error") is not None
    clase, criterio = CRITERIOS.get(nombre, ("—", "—"))
    filas.append({"Clase o módulo": clase, "Criterio de aceptación": criterio, "Prueba": nombre,
                  "Tiempo (s)": round(float(caso.get("time")), 3), "Resultado": "FALLÓ" if fallo else "PASÓ"})
df = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", 60)
display(df)
print(f"\nPasaron {(df['Resultado'] == 'PASÓ').sum()} de {len(df)} pruebas.")

## 7. Mutaciones reales y marcador baja · Figura 12 · Tablas 18 y 20 📸
Aplica cada mutación, corre pytest con el archivo mutado y lo restaura. Resultado esperado: **3 de 4 detectadas**; MUT-01 sobrevive porque ninguna prueba usa costo 0.

In [ ]:
import datetime
import pathlib
import subprocess

import pandas as pd

print("===========================================================")
print("   FIGURA 12 · TABLAS 18 Y 20: MUTACIONES REALES Y MARCADOR BAJA")
print("FECHA Y HORA DE EJECUCIÓN:", datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S"))
print("===========================================================\n")

# (id, archivo, descripción, texto original, texto mutado)
MUTACIONES = [
    ("MUT-01", "tests/test_modelos.py",
     "Cambiar '>' por '>=' en la validación de costo_base (aceptar costo 0)",
     "if costo_base <= 0:", "if costo_base < 0:"),
    ("MUT-02", "tests/test_modelos.py",
     "Alterar las transiciones válidas: permitir Registrado → Entregado",
     '"Registrado": ["Validado", "Cancelado"]', '"Registrado": ["Validado", "Cancelado", "Entregado"]'),
    ("MUT-03", "tests/test_gestores.py",
     "Omitir la verificación de id duplicado al agregar",
     'if ruta_dict["id"] in self.rutas:', "if False:"),
    ("MUT-04", "tests/test_carga_lote.py",
     "Modificar el filtro del lote para aceptar costos negativos o nulos",
     "if costo <= 0: raise ValueError()", "if costo < -10**9: raise ValueError()"),
]


def correr_pytest():
    r = subprocess.run(["pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
    ultima = [l for l in r.stdout.splitlines() if l.strip()][-1]
    return r.returncode, ultima


codigo, resumen = correr_pytest()
print("Suite sin mutaciones:", resumen)
assert codigo == 0, "La suite debe pasar completa antes de mutar."

filas = []
for ident, archivo, descripcion, original, mutado in MUTACIONES:
    ruta = pathlib.Path(archivo)
    texto = ruta.read_text(encoding="utf-8")
    if original not in texto:
        filas.append({"ID": ident, "Archivo": archivo, "Modificación": descripcion,
                      "Resultado de pytest": "No aplicada (no se encontró el texto)", "Estado": "—"})
        continue
    ruta.write_text(texto.replace(original, mutado, 1), encoding="utf-8")
    try:
        codigo, resumen = correr_pytest()
    finally:
        ruta.write_text(texto, encoding="utf-8")  # siempre se restaura el archivo original
    filas.append({"ID": ident, "Archivo": archivo, "Modificación": descripcion,
                  "Resultado de pytest": resumen,
                  "Estado": "Detectada (alguna prueba falló)" if codigo != 0 else "SOBREVIVIÓ (ninguna prueba falló)"})

df = pd.DataFrame(filas)
detectadas = (df["Estado"].str.startswith("Detectada")).sum()
print(f"\n--> Mutaciones detectadas: {detectadas} de {len(df)} ({detectadas * 100 // len(df)} %)\n")
display(df)

codigo, resumen = correr_pytest()
print("\nSuite restaurada:", resumen)

print("\n--> Pruebas de prioridad BAJA:")
!pytest -v -m baja


## 8. Cobertura de la matriz de criterios · Tabla 16 📸 (apoyo)
Calcula las cifras de la sección 3.2 a partir de la matriz del reporte. `*` = prueba sobre una clase simplificada definida en el propio archivo de prueba.

In [ ]:
import pandas as pd
encabezado("TABLA 16: COBERTURA DE LA MATRIZ DE CRITERIOS")
ALTA = {"T-01","T-02","T-03","T-04","T-06","T-08","T-09","T-10","T-12","T-17","T-21","T-23","T-25"}
PRUEBAS = {
    "T-01": ("test_transicion_estado_valida*", "Parcial"),
    "T-02": ("test_transicion_estado_invalida*", "Parcial"),
    "T-08": ("test_agregar_ruta_duplicada*", "Parcial"),
    "T-21": ("test_cargar_archivo_inexistente*", "Contradice"),
    "T-22": ("test_procesar_lote_con_filas_invalidas*", "Parcial"),
}
filas = []
for i in range(1, 26):
    t = f"T-{i:02d}"
    prueba, res = PRUEBAS.get(t, ("Sin prueba", "—"))
    filas.append({"Criterio": t, "Prioridad": "Alta" if t in ALTA else "Media/Baja", "Prueba": prueba, "Resultado": res})
df = pd.DataFrame(filas)
display(df)
parciales = (df["Resultado"] == "Parcial").sum()
alta_sin = df[(df["Prioridad"] == "Alta") & (df["Resultado"].isin(["—", "Contradice"]))]
print(f"Criterios: {len(df)} (alta: {len(ALTA)})")
print(f"Con prueba que ejecuta el código documentado: 0 de {len(df)} (0 %)")
print(f"Con prueba parcial sobre el modelo simplificado: {parciales} de {len(df)} ({parciales * 100 // len(df)} %)")
print(f"Con prueba que contradice el criterio: {(df['Resultado'] == 'Contradice').sum()}")
print(f"Alta sin ninguna prueba válida: {len(alta_sin)} → {', '.join(alta_sin['Criterio'])}")

## 9. Árbol real del repositorio · Figura 4 📸
Clona el repositorio del equipo y muestra sus archivos. Si el repositorio es privado, pedirá acceso y fallará: en ese caso, hazlo público o agrega al profesor como colaborador.

In [ ]:
import os
encabezado("FIGURA 4: ÁRBOL DE ARCHIVOS DEL REPOSITORIO")
REPO = "https://github.com/angeldetre12-gif/rutas-Bajio-2p.git"
!rm -rf repo_equipo && git clone -q {REPO} repo_equipo
if os.path.isdir("repo_equipo"):
    !apt-get -qq install -y tree > /dev/null 2>&1
    !cd repo_equipo && (tree -a -I ".git|__pycache__" . 2>/dev/null || find . -path ./.git -prune -o -type f -print | sort)
    archivos = {f for raiz, _, fs in os.walk("repo_equipo") if ".git" not in raiz for f in fs}
    faltan = [m for m in ["modelos.py", "gestores.py", "carga_lote.py", "excepciones.py"] if m not in archivos]
    print("\nMódulos del sistema que no están en el repositorio:", ", ",.join(faltan) if faltan else "ninguno, ya están todos")
else:
    print("No se pudo clonar el repositorio (¿es privado?).")

## 10. Historial de Git · Figura 13 · Tabla 24 📸
Muestra todos los commits con autor y fecha, y cuántos hizo cada integrante. La guía pide aportes de todos y un commit final con el mensaje `v-p2 - entrega final`.

In [ ]:
import os
encabezado("FIGURA 13 · TABLA 24: HISTORIAL DE COMMITS DEL REPOSITORIO")
if os.path.isdir("repo_equipo"):
    print("--- Commits (todas las ramas) ---")
    !cd repo_equipo && git log --all --date=format:"%d/%m/%Y %H:%M" --pretty=format:"%h | %ad | %an | %s"
    print("\n\n--- Commits por integrante ---")
    !cd repo_equipo && git shortlog -sn --all
    print("\n--- Commit final «v-p2 - entrega final» ---")
    !cd repo_equipo && (git log --all --oneline --grep="v-p2 - entrega final" | grep . || echo "Todavía no existe.")
else:
    print("Primero ejecuta la celda 9 (clonar el repositorio).")

## Pendientes que no salen de este cuaderno
- **Figuras 5 y 6:** ejecutar `cargar_lote` del sistema real con `lote_ejemplo.csv` y mostrar `rechazos_ejemplo.csv` (código de Luis).
- **Subir al repositorio** los módulos del sistema (`modelos.py`, `gestores.py`, `carga_lote.py`, `excepciones.py`), con commits de cada integrante.
- **Commit final:** `git commit -m "v-p2 - entrega final"` y `git push`.
- **Actas (Anexo B), bitácoras (Anexo E), firmas (Tabla A2), fecha de entrega y una fuente no IA por integrante.**